# Multilayer Perceptron From Scratch

forward pass    →  compute loss
backward pass   →  compute gradients   (fills .grad on every parameter)
update step     →  parameters -= lr * parameter.grad   ← THIS is the learning
zero gradients  →  parameter.grad = 0   (reset before next iteration)

**What is learned and persists?** the parameters
- Parameters = the network's memory. They define what the model knows. They persist across batches, epochs, deployment.

**What get recomputed fresh every iteration?** the gradient
- Gradients = a one-time directional signal saying "to reduce the loss on this batch, nudge w in this direction by this much." They're a tool, used once, then discarded (zeroed).

In [80]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [81]:
from graphviz import Digraph


def trace(root):
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format="svg", graph_attr={"rankdir": "LR"})  # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))

        # for any value in graph, create a rectangular node for it
        dot.node(
            name=uid,
            label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad),
            shape="record",
        )
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name=uid + n._op, label=n._op)

            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [82]:
class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None  # by default, we do nothing
        self._prev = set(_children)
        self._op = _op  # keep track of the operation
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        # gradient
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return (-self) + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def __pow__(self, other):
        assert isinstance(
            other, (int, float)
        ), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f"**{other}")

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward

        return out

    def backward(self):
        # build graph
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
f = Value(-2.0, label="f")
L = d * f  # output of graph
L.label = "L"
print(d)

Value(data=4.0)


In [83]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]  # random weight
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # w * x + b

        # forward pass of the neuron
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

# Test With Simple Dataset & Build A Training Loop
We want the neural net to predict these example. The trick is to calculate the loss and optimize for the lost to be 0. This means that the knowledge is stored in the gradient

In [84]:
# Build an MLP
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

Value(data=0.6892644551367911)

In [85]:
# Dataset
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]  # desired targets

In [86]:
# Training Loop
for k in range(100):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))

    # flush gradient
    for p in n.parameters():
        p.grad = 0.0

    # calculate gradient based on the loss via backward pass
    loss.backward()

    # update the weight & bias via gradient descent
    for p in n.parameters():
        p.data += -0.05 * p.grad  # step size to minimize the loss

    print(k, loss.data)

0 6.46828012310676
1 3.6719556640293383
2 1.9369335233887597
3 0.7541862805568932
4 0.4321414923665104
5 0.19064896949450322
6 0.14308109115802628
7 0.11530450062988759
8 0.09608055479106589
9 0.08207404888986572
10 0.0714715588100097
11 0.06319829694212933
12 0.056580668070411386
13 0.05117744603413185
14 0.04668883823828621
15 0.0429047387705916
16 0.039673927633033956
17 0.03688502511531441
18 0.03445432062342234
19 0.03231776730046777
20 0.030425580948335237
21 0.02873851259194708
22 0.027225223393629053
23 0.02586040185266981
24 0.02462339090503131
25 0.023497171687536088
26 0.02246760093031519
27 0.021522831449397865
28 0.02065286666760385
29 0.01984921450348
30 0.01910461580512193
31 0.018412829321114075
32 0.017768459988350273
33 0.017166820722980678
34 0.016603820353706864
35 0.016075872122859745
36 0.015579818495042603
37 0.015112868989788168
38 0.014672548487137288
39 0.014256654009138829
40 0.013863218402878835
41 0.01349047967546266
42 0.013136854982861
43 0.01280091847059

In [87]:
ypred

[Value(data=0.9691216318654663),
 Value(data=-0.9706501875036446),
 Value(data=-0.9562244440430792),
 Value(data=0.961859062194536)]